# Exploratory Data Analysis (EDA)
In this notebook, we will explore the PlayHack ML track dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style="whitegrid")



## 1. Load Data


In [ ]:
# Load labels and metadata
labels = pd.read_csv('train_labels.csv')
metadata = pd.read_csv('athlete_metadata.csv')

# Load daily/hourly data
daily_activity = pd.read_csv('dailyActivity_merged.csv')
sleep_day = pd.read_csv('sleepDay_merged.csv')
weight_log = pd.read_csv('weightLogInfo_merged.csv')
training_sessions = pd.read_csv('training_sessions.csv')

hourly_steps = pd.read_csv('hourlySteps_merged.csv')
hourly_calories = pd.read_csv('hourlyCalories_merged.csv')
hourly_intensities = pd.read_csv('hourlyIntensities_merged.csv')
hourly_heartrate = pd.read_csv('hourlyHeartrate_merged.csv')

print(f"Labels shape: {labels.shape}")
print(f"Metadata shape: {metadata.shape}")



## 2. Target Variable Analysis


In [ ]:
# Distribution of injury (Task A target)
fig = px.pie(labels, names='injured_in_risk_window', title='Proportion of Injured Athletes in Risk Window')
fig.show()



In [ ]:
# Distribution of onset_day_offset and recovery_duration (Task B targets)
injured_labels = labels[labels['injured_in_risk_window'] == 1]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(injured_labels['onset_day_offset'], bins=15, kde=True)
plt.title('Distribution of Onset Day Offset')
plt.xlabel('Day in Risk Window (31-60)')

plt.subplot(1, 2, 2)
sns.histplot(injured_labels['recovery_duration'], bins=20, kde=True, color='orange')
plt.title('Distribution of Recovery Duration')
plt.xlabel('Days of Recovery')
plt.tight_layout()
plt.show()



## 3. Metadata Analysis


In [ ]:
# Merge labels with metadata
df_meta = pd.merge(metadata, labels, on='athlete_id', how='left')
df_meta.head()



In [ ]:
# Injury rate by sport
plt.figure(figsize=(10, 6))
sns.barplot(data=df_meta, x='sport', y='injured_in_risk_window', ci=None)
plt.title('Injury Rate by Sport')
plt.ylabel('Proportion Injured')
plt.xticks(rotation=45)
plt.show()



In [ ]:
# Numeric features distribution vs Injury
numeric_cols = ['age', 'height_cm', 'weight_kg_baseline', 'years_playing', 'prior_season_injury_count']
plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(data=df_meta, x='injured_in_risk_window', y=col)
    plt.title(f'{col} by Injury Status')
plt.tight_layout()
plt.show()



In [ ]:
# Correlation Heatmap for metadata
plt.figure(figsize=(10, 8))
corr = df_meta[numeric_cols + ['injured_in_risk_window', 'onset_day_offset', 'recovery_duration']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap - Metadata & Targets')
plt.show()



## 4. Daily Activity & Sleep Analysis


In [ ]:
# Ensure common ID column name
daily_activity.rename(columns={'Id': 'athlete_id'}, inplace=True)
sleep_day.rename(columns={'Id': 'athlete_id'}, inplace=True)

# Aggregate daily activity per athlete
activity_agg = daily_activity.groupby('athlete_id').agg({
    'TotalSteps': 'mean',
    'TotalDistance': 'mean',
    'VeryActiveMinutes': 'mean',
    'SedentaryMinutes': 'mean',
    'Calories': 'mean'
}).reset_index()

df_activity = pd.merge(df_meta[['athlete_id', 'injured_in_risk_window']], activity_agg, on='athlete_id', how='inner')

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
sns.kdeplot(data=df_activity, x='TotalSteps', hue='injured_in_risk_window', fill=True)
plt.title('Average Daily Steps Distribution')

plt.subplot(1, 2, 2)
sns.kdeplot(data=df_activity, x='VeryActiveMinutes', hue='injured_in_risk_window', fill=True)
plt.title('Average Very Active Minutes')
plt.tight_layout()
plt.show()



In [ ]:
# Sleep efficiency
sleep_day['SleepEfficiency'] = sleep_day['TotalMinutesAsleep'] / sleep_day['TotalTimeInBed']
sleep_agg = sleep_day.groupby('athlete_id').agg({
    'TotalMinutesAsleep': 'mean',
    'SleepEfficiency': 'mean'
}).reset_index()

df_sleep = pd.merge(df_meta[['athlete_id', 'injured_in_risk_window']], sleep_agg, on='athlete_id', how='inner')

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_sleep, x='injured_in_risk_window', y='SleepEfficiency')
plt.title('Sleep Efficiency by Injury Status')
plt.show()



## 5. Training Sessions Analysis


In [ ]:
# Calculate duration of sessions
training_sessions['duration'] = training_sessions['end_hour'] - training_sessions['start_hour']
# Handle negative duration if crossing midnight (assuming 24h format)
training_sessions['duration'] = training_sessions['duration'].apply(lambda x: x if x >= 0 else x + 24)

train_agg = training_sessions.groupby('athlete_id').agg({
    'session_id': 'count',
    'duration': 'sum'
}).rename(columns={'session_id': 'total_sessions', 'duration': 'total_training_hours'}).reset_index()

df_train = pd.merge(df_meta[['athlete_id', 'injured_in_risk_window']], train_agg, on='athlete_id', how='left').fillna(0)

fig = px.scatter(df_train, x='total_sessions', y='total_training_hours', color='injured_in_risk_window', 
                 title='Total Sessions vs Total Training Hours', opacity=0.7)
fig.show()



## 6. Time Series / Hourly Data (Example)


In [ ]:
# Merge hourly HR and Steps
hourly_heartrate.rename(columns={'Id': 'athlete_id'}, inplace=True)
hourly_steps.rename(columns={'Id': 'athlete_id'}, inplace=True)

# Average heart rate over the day (0-23 hours) - simplified extraction from ActivityHour
try:
    hourly_heartrate['Hour'] = pd.to_datetime(hourly_heartrate['ActivityHour']).dt.hour
    hr_hourly = hourly_heartrate.groupby('Hour')['AvgHeartRate'].mean().reset_index()
    
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=hr_hourly, x='Hour', y='AvgHeartRate', marker='o')
    plt.title('Average Heart Rate Pattern Across the Day')
    plt.xticks(range(24))
    plt.show()
except Exception as e:
    print("Could not parse datetime, check format:", e)

